# Script to Append Census Tract Indicators, Social Vulnerability Index data, and USDA Food Access Research Atlas data

This code builds off of the "a-merging_restauranthealthinspections.ipynb" notebook. 

After merging the restaurant data, we sent it to the company Geocodio to geocode the dataset, appending latitude, longitude, and other geographic variables. Therefore, we don't have the code that geocoded that data, but the geocoded restaurant health inspections dataset can be found at the following path: "IDS705_TeamPandas/Data/Geocoded_Data/healthinspections_2024_geocoded.csv".

Description:
- This script creates a 2 mile buffer around each restaurant, and identifies the Census tracts that intersect with this buffer.
- Then, we download a list of variables from the 2023 5-Year American Community Survey dataset using the `censusdis` package. These variables are: `"MEDIAN_HOUSEHOLD_INCOME_VARIABLE", "AVG_HOUSEHOLD_SIZE", "TOTAL_POPULATION", "MEDIAN GROSS RENT"`
- Grouping by restaurant, we calculate the average value of each of these Census variables for each intersecting Census tract. Then we merge these back in to the restaurants dataset.  
- Next, we merge in the Social Vulnerability Index data at the Census tract level. 
- Finally, we merge in the USDA Food Access Research Atlas data at the Census tract level.


Datasets are located in the "Data" folder of this repository:
- "healthinspections_2024_geocoded.csv" 
    - Geocoded restaurant health inspections dataset for select counties in CA, TX, KY for the year 2024. 

Other Datasets are used in this script, but are too large to include in the GitHub repository. Instead, I stored them in a "LargeData" folder that I hid by including the folder name in the `.gitignore` file.

These large datasets are:
- "FoodAccessResearchAtlasData2019.xlsx"
    - Downloaded from: https://www.ers.usda.gov/data-products/food-access-research-atlas/download-the-data 
- "SVI_2022_US.csv"
    - Downloaded at the Census tract level from https://www.atsdr.cdc.gov/place-health/php/svi/svi-data-documentation-download.html 
- tract_shapefiles
    - These are the shapefiles for Census Tract. These were downloaded from the U.S. Census Bureau at https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html 
    - The Census Bureau only allows you to download these by state so I downloaded the shapefiles for California, Kentucky, and Texas and concatenated them into the same dataset.



In [1]:
#imports
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import censusdis.data as ced
from censusdis.states import ALL_STATES_AND_DC

# Appending Census Datasets to Geocoded Restaurant Health Inspections Dataset


In [3]:
#loading geocoded restaurant health inspection data
df = pd.read_csv("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Data/Geocoded_Data/healthinspections_2024_geocoded.csv")

/var/folders/81/w_61xz297rv4ggdktb58tlxm0000gn/T/ipykernel_87965/2102467681.py:2: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Data/Geocoded_Data/healthinspections_2024_geocoded.csv")


In [4]:
#convert dataframe into GeoDataFrame

#Creating geometry column
geometry = gpd.points_from_xy(df.Longitude, df.Latitude)
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs="EPSG:4326")  # WGS84


In [5]:
# Reading in Census tract boundaries from shapefiles
CAtract_gdf = gpd.read_file("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/LargeData/tract_shapefiles/tl_2024_06_tract.shp")
KYtract_gdf =  gpd.read_file("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/LargeData/tract_shapefiles/tl_2024_06_tract.shp")
TXtract_gdf =  gpd.read_file("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/LargeData/tract_shapefiles/tl_2024_48_tract.shp")

combined_census_gdf = pd.concat([CAtract_gdf, KYtract_gdf, TXtract_gdf], ignore_index=True)

# projecting Census tract shapefile to a metric CRS (for distance calculation)
censustract_gdf = combined_census_gdf.to_crs("EPSG:3857")  # Web Mercator: units in meters
restaurants_gdf = gdf.to_crs("EPSG:3857")

# Create a small buffer around each restaurant to avoid precision issues on borders
restaurants_gdf['buffer'] = restaurants_gdf.geometry.buffer(0.001)  # Small buffer

# Perform spatial join with buffered geometries
restaurants_with_tracts = gpd.sjoin(restaurants_gdf, censustract_gdf, how="left", predicate="within")

# Drop duplicates based on the restaurant's geometry
restaurants_with_tracts = restaurants_with_tracts.drop_duplicates(subset='geometry')

# Rename the 'GEOID' column to 'PRIMARY_CENSUSTRACT'
restaurants_with_tracts = restaurants_with_tracts.rename(columns={"GEOID": "PRIMARY_CENSUSTRACT"})

In [6]:
#downloading census data

#Variables we want to query
# Variables = ["MEDIAN_HOUSEHOLD_INCOME_VARIABLE", "AVG_HOUSEHOLD_SIZE", "TOTAL_POPULATION", "MEDIAN GROSS RENT"]
VARIABLES = ["NAME","B19013_001E", "B25010_001E", "B01003_001E", "B25064_001E"]

ACS2023_indicators = ced.download(
    dataset="acs/acs5",
    vintage=2023,
    download_variables=VARIABLES,
    state=ALL_STATES_AND_DC,
    tract="*",
    with_geometry=True
)

In [7]:
#merge ACS 2023 indicators into censustract_gdf
ACS2023_indicators = ACS2023_indicators.rename(columns={"B19013_001E": "MEDIAN_INCOME", "B25010_001E": "AVG_HOUSEHOLD_SIZE", "B01003_001E": "TOTAL_POPULATION", "B25064_001E": "MEDIAN_GROSS_RENT"})
censustract_gdf = censustract_gdf.merge(ACS2023_indicators[['TRACT', 'MEDIAN_INCOME', 'AVG_HOUSEHOLD_SIZE', 'TOTAL_POPULATION', 'MEDIAN_GROSS_RENT']], left_on="TRACTCE", right_on="TRACT", how="left")

In [8]:
#For each restaurant, find nearby Census tracts within 2 miles and average
from shapely.geometry import Point

# Remove any existing 'index_right' column to prevent conflict
if 'index_right' in restaurants_with_tracts.columns:
    restaurants_with_tracts = restaurants_with_tracts.drop(columns='index_right')

# Convert 2 miles to meters
buffer_radius = 3218.69  # meters

# Create buffer around each restaurant point
restaurants_with_tracts['buffer'] = restaurants_with_tracts.geometry.buffer(buffer_radius)

# Use spatial join: buffered restaurant areas to intersecting TRACTS
buffered_join = gpd.sjoin(
    gpd.GeoDataFrame(restaurants_with_tracts, geometry='buffer'),
    censustract_gdf[['TRACT', 'geometry', 'MEDIAN_INCOME', 'AVG_HOUSEHOLD_SIZE', 'TOTAL_POPULATION', 'MEDIAN_GROSS_RENT']],
    how='left',
    predicate='intersects'
)

buffered_join = buffered_join.reset_index()

# Group by restaurant and calculate average income of surrounding Census tracts
avg_censusindicators_nearby = buffered_join.groupby('index').agg(
    AVG_INCOME_NEARBY_TRACTS=('MEDIAN_INCOME', 'mean'),
    AVG_HH_SIZE_NEARBY_TRACTS=('AVG_HOUSEHOLD_SIZE', 'mean'),
    AVG_RENT_NEARBY_TRACTS=('MEDIAN_GROSS_RENT', 'mean'),
    AVG_POP_NEARBY_TRACTS=('TOTAL_POPULATION', 'mean')
)

# Merge the average income back into the original restaurant dataframe
restaurants_final = restaurants_with_tracts.join(avg_censusindicators_nearby)

In [ ]:
#Export just the geocoded restaurant dataset with census indicators (if needed)
# output_path = "/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Output/healthinspections_2024_censusindicators_censustract.csv"
# restaurants_final.to_csv(output_path, index=False)

# Adding SVI Data at the Census tract Level


In [9]:
#loading CDC Social Vulnerability Index data that is downloaded at the Census Tract level
states_of_interest = ['CA', 'KY', 'TX']

svitract_df = pd.read_csv("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/LargeData/SVI_2022_US.csv")
svitract_df = svitract_df[svitract_df['ST_ABBR'].isin(states_of_interest)]
#add leading zero to geoids where that was dropped by excel format
svitract_df['FIPS'] = svitract_df['FIPS'].astype(str).str.zfill(11)
svitract_df['FIPS'] = svitract_df['FIPS'].astype(str)

In [10]:
#left outerjoin with restaurant data
restaurants_withsvi = restaurants_final.merge(svitract_df, left_on="PRIMARY_CENSUSTRACT", right_on="FIPS", how="left")


In [11]:
#drop unneeded columns
svicols_todrop = ["ST", "STATE_x", "STATE_y", "STCNTY", "COUNTY", "FIPS", "M_TOTPOP", "M_HU", "M_HH", "M_POV150",
                  'M_UNEMP',  'M_HBURD', 'M_NOHSDP', 'M_UNINSUR', 'M_AGE65', 'M_AGE17', 
                  'M_DISABL', 'M_SNGPNT', 'M_LIMENG',  'M_MINRTY','M_MUNIT', 'M_MOBILE', 
                  'M_CROWD', 'M_NOVEH', 'M_GROUPQ', 'MP_POV150', 'MP_UNEMP', 
                  'MP_HBURD', 'MP_NOHSDP', 'MP_UNINSUR', 'MP_AGE65', 'MP_AGE17', 
                  'MP_DISABL', 'MP_SNGPNT', 'MP_LIMENG', 'MP_MINRTY', 'MP_MUNIT',
                    'MP_MOBILE', 'MP_CROWD', 'MP_NOVEH', 'MP_GROUPQ', 'M_NOINT', 
                    'M_AFAM', 'M_HISP', 'M_ASIAN', 'M_AIAN', 'M_NHPI', 'M_TWOMORE', 
                    'M_OTHERRACE','MP_NOINT', 'MP_AFAM', 'MP_HISP', 'MP_ASIAN', 
                    'MP_AIAN', 'MP_NHPI', 'MP_TWOMORE', 'MP_OTHERRACE']
restaurants_withsvi = restaurants_withsvi.drop(columns=svicols_todrop)

# Merging Food Access Research Atlas Data


In [13]:
usda_foodaccessdf = pd.read_excel("/Users/pdeguz01/Documents/git/IDS705_TeamPandas/LargeData/FoodAccessResearchAtlasData2019.xlsx", sheet_name=2)
states_needed = ['California', 'Kentucky', 'Texas']
usda_foodaccessdf = usda_foodaccessdf[usda_foodaccessdf['State'].isin(states_needed)]

In [14]:
usda_colstodrop = ["State", "County"]
usda_foodaccessdf = usda_foodaccessdf.drop(columns=usda_colstodrop)
#add leading zero to geoids where that was dropped by excel format
usda_foodaccessdf['CensusTract'] = usda_foodaccessdf['CensusTract'].astype(str).str.zfill(11)
usda_foodaccessdf['CensusTract'] = usda_foodaccessdf['CensusTract'].astype(str)
usda_foodaccessdf = usda_foodaccessdf.rename(columns=lambda col: f"USDA_{col}")

In [15]:
#merge USDA Food Access data with restaurant dataset
restaurants_withsvifoodaccess = restaurants_withsvi.merge(usda_foodaccessdf, left_on="PRIMARY_CENSUSTRACT", right_on="USDA_CensusTract", how="left")

In [16]:
# count rows by state
state_count2 = restaurants_withsvifoodaccess["State"].value_counts()
print(state_count2)

State
CA    22427
KY     3236
TX     3026
Name: count, dtype: int64


In [17]:
# Export dataset
output_path_final = "/Users/pdeguz01/Documents/git/IDS705_TeamPandas/Data/Merged_Data/restauranthealthinspections2024_CensusSVIUSDA.csv"
restaurants_withsvifoodaccess.to_csv(output_path_final, index=False)